In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os
from scipy.stats import gaussian_kde
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
from matplotlib import colors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
lat = np.arange(-40, 41, 1)
lon = np.arange(0, 361, 1)

TC_path1d = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/genesis-new-1d/'
TC_path2d = r'../../../study/data/noaa-20c/NOAA_20C_TXT_30_all_bing/genesis-new-2d/'
TCT_path = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/'
TCT_path_grid = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TCT/'
soda_path = r'../../../study/data/soda/'
path_noaa = r'../../../study/data/noaa-20c/'

In [ ]:
def composite_ttest(ssta_sum, event, single_sample=False):
    mask = ssta_sum.time.dt.year.isin(event.astype('datetime64[Y]').astype(str).astype(int))
    ssta_mar = ssta_sum.where(mask, drop=True)
    if single_sample:
        n = ssta_mar.shape[0]
        comp = ssta_mar.mean(dim='time')
        std = ssta_mar.std(dim='time', ddof=1)
        t_stat = comp / std *np.sqrt(n)
        p_value = 2*stats.t.sf(np.abs(t_stat), n)
        return comp, p_value

    ssta_nor = ssta_sum.where(~mask, drop=True)

    n1 = ssta_mar.shape[0]
    n2 = ssta_nor.shape[0]

    m1 = ssta_mar.mean(dim='time')
    m2 = ssta_nor.mean(dim='time')
    v1 = ssta_mar.var(dim='time', ddof=1)
    v2 = ssta_nor.var(dim='time', ddof=1)

    t_stat = (m1 - m2)/np.sqrt(v1/n1 + v2/n2)
    df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
    p_value = stats.t.sf(np.abs(t_stat), df)
    p_value = xr.DataArray(p_value, coords=t_stat.coords)
    return m1, p_value

def composite_tctest(tct_mar):
    #(case, lat, lon)
    n = tct_mar.shape[0]
    comp = tct_mar.mean(axis=0)
    std = tct_mar.std(axis=0)
    t_stat = comp / std*np.sqrt(n)
    p_value = 2*stats.t.sf(np.abs(t_stat), n)
    return comp, p_value

def composite_ttest_time(ssta_hov, event):
    mask = ssta_hov.year.isin(event.astype('datetime64[Y]').astype(str).astype(int))

    ssta_mar = ssta_hov.where(mask, drop=True)
    ssta_nor = ssta_hov.where(~mask, drop=True)

    n1 = ssta_mar.shape[0]
    n2 = ssta_nor.shape[0]

    m1 = ssta_mar.mean(dim='year')
    m2 = ssta_nor.mean(dim='year')
    v1 = ssta_mar.var(dim='year', ddof=1)
    v2 = ssta_nor.var(dim='year', ddof=1)

    t_stat = (m1 - m2)/np.sqrt(v1/n1 + v2/n2)
    df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
    p_value = stats.t.sf(np.abs(t_stat), df)
    p_value = xr.DataArray(p_value, coords=t_stat.coords)
    return m1, p_value

In [ ]:
d = 6.371e+6 * np.pi / 180
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2014-01')]
ep = ep[ep < np.datetime64('2014-01')]
mix = mix[mix < np.datetime64('2014-01')]
suc = suc[suc < np.datetime64('2014-01')]
events = [mix, cp, ep, suc]
lat = np.arange(-40, 41, 1)
lon = np.arange(0, 360, 1)
nx, ny = len(lon), len(lat)
ssta = xr.open_dataset('./data/ssta.nc').sst.loc['1900-01':'2014-12', -40:40, 0:359]
apcp1 = xr.open_dataset(path_noaa+'apcp.mon.mean.recg.nc').__xarray_dataarray_variable__.loc[:, -40:40, 0:359]
uwnd = xr.open_dataset(path_noaa+'uwnd.mon.mean.tp.nc').uwnd.loc[:, [850, 200]]
vwnd = xr.open_dataset(path_noaa+'vwnd.mon.mean.tp.nc').vwnd.loc[:, [850, 200]]
tct = xr.open_dataset('./data-20c/NOAA-20C_TCT_grid.nc').tct_grid.loc['1900-01':'2014-12', -40:40, 0:359]
kde = xr.open_dataset('./data-20c/kde.nc').kde.assign_coords({'time':pd.date_range('1900-01', '2014-12', freq='MS')}).loc[:, :, 0:359]
vort = (vwnd.loc[:, 850].differentiate('lon')  - uwnd.loc[:, 850].differentiate('lat'))/ d
omega = xr.open_dataset(path_noaa+'omega.mon.mean.tp.nc').omega.loc[:, 500]
dudz = uwnd.loc[:, 200] - uwnd.loc[:, 850]
dvdz = vwnd.loc[:, 200] - vwnd.loc[:, 850]

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

def process_space(ssta):
    ssta -= ssta.groupby('time.month').map(rolling)
    ssta_sum = ssta.sel(time=ssta.time.dt.month.isin([6, 7, 8, 9, 10, 11])).coarsen(time=6).mean()
    ssta_win = ssta[11:-1].sel(time=ssta[11:-1].time.dt.month.isin([1, 2, 3, 4, 5, 12])).coarsen(time=6).mean()
    return ssta, ssta_sum, ssta_win

ssta, ssta_sum, ssta_win = process_space(ssta)
kde, kde_sum, kde_win = process_space(kde)
tct, tct_sum, tct_win = process_space(tct)
vort, vort_sum, vort_win = process_space(vort)
dudz, dudz_sum, dudz_win = process_space(dudz)
omega, omega_sum, omega_win = process_space(omega)
uwnd, uwnd_sum, uwnd_win = process_space(uwnd.sel(level=850))

In [ ]:
da = vort
month = da.time.dt.month

lat_nh = (da.lat >= 5) & (da.lat <= 25) 
lat_sh = (da.lat >= -25) & (da.lat <= -5)
is_jun_nov = (month >= 6) & (month <= 11)
is_dec_may = (month >= 12) | (month <= 5)

mask = (is_jun_nov & lat_nh) | (is_dec_may & lat_sh)

vort_hov = vort.where(mask).mean('lat')
kde_hov = kde.where(mask).mean('lat')
dudz_hov = dudz.where(mask).mean('lat')
apcp_hov = apcp1.where(mask).mean('lat')

vort_hov = vort_hov.assign_coords(
    year=vort_hov.time.dt.year,
    month=vort_hov.time.dt.month
).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')
kde_hov = kde_hov.assign_coords(
    year=kde_hov.time.dt.year,
    month=kde_hov.time.dt.month
).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')
dudz_hov = dudz_hov.assign_coords(
    year=dudz_hov.time.dt.year,
    month=dudz_hov.time.dt.month
).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')
apcp_hov = apcp_hov.assign_coords(
    year=apcp_hov.time.dt.year,
    month=apcp_hov.time.dt.month
).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')

In [ ]:
month = np.arange(1, 13)
nt, nx = len(month), len(lon)
ds_nino = xr.Dataset(
    {
        'vort_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'kde_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'dudz_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'apcp_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'vort_p':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'kde_p':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'dudz_p':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'apcp_p':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'CP', 'EP', 'SUC']),
        'stage':(['stage'], ['current', 'next', 'next2']),
        'month':(['month'], month),
        'lon':(['lon'], lon)
    }
)

for i in range(4):
    event = events[i]
    for j in range(3):
        ds_nino.vort_hov[i, j, :, :], ds_nino.vort_p[i, j, :, :] = composite_ttest_time(vort_hov, event+np.timedelta64(j, 'Y'))
        ds_nino.kde_hov[i, j, :, :], ds_nino.kde_p[i, j, :, :] = composite_ttest_time(kde_hov, event+np.timedelta64(j, 'Y'))
        ds_nino.dudz_hov[i, j, :, :], ds_nino.dudz_p[i, j, :, :] = composite_ttest_time(dudz_hov, event+np.timedelta64(j, 'Y'))
        ds_nino.apcp_hov[i, j, :, :], ds_nino.apcp_p[i, j, :, :] = composite_ttest_time(apcp_hov, event+np.timedelta64(j, 'Y'))

ds_nino.to_netcdf('./significant/composite_vort_hov.nc')

In [ ]:
ny, nx = len(lat), len(lon)
ds_nino = xr.Dataset(
    {
        'ssta':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'kde':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'tct':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'vort':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'dudz':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'omega':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'uwnd':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'ssta_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'kde_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'tct_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'vort_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'dudz_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'omega_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
        'uwnd_p':(['type', 'season', 'stage', 'lat', 'lon'], np.empty((4, 2, 2, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'CP', 'EP', 'SUC']),
        'season':(['season'], ['sum', 'win']), 
        'stage':(['stage'], ['current', 'next']),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

single = False
for i in range(4):
    event = events[i]
    ds_nino.ssta[i, 0, 0, :, :], ds_nino.ssta_p[i, 0, 0, :, :] = composite_ttest(ssta_sum, event, single_sample=single)
    ds_nino.ssta[i, 1, 0, :, :], ds_nino.ssta_p[i, 1, 0, :, :] = composite_ttest(ssta_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.ssta[i, 0, 1, :, :], ds_nino.ssta_p[i, 0, 1, :, :] = composite_ttest(ssta_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.ssta[i, 1, 1, :, :], ds_nino.ssta_p[i, 1, 1, :, :] = composite_ttest(ssta_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    ds_nino.kde[i, 0, 0, :, :], ds_nino.kde_p[i, 0, 0, :, :] = composite_ttest(kde_sum, event, single_sample=single)
    ds_nino.kde[i, 1, 0, :, :], ds_nino.kde_p[i, 1, 0, :, :] = composite_ttest(kde_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.kde[i, 0, 1, :, :], ds_nino.kde_p[i, 0, 1, :, :] = composite_ttest(kde_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.kde[i, 1, 1, :, :], ds_nino.kde_p[i, 1, 1, :, :] = composite_ttest(kde_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    # ds_nino.tct[i, 0, 0, :, :], ds_nino.tct_p[i, 0, 0, :, :] = composite_ttest(tct_sum, event, single_sample=single)
    # ds_nino.tct[i, 1, 0, :, :], ds_nino.tct_p[i, 1, 0, :, :] = composite_ttest(tct_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    # ds_nino.tct[i, 0, 1, :, :], ds_nino.tct_p[i, 0, 1, :, :] = composite_ttest(tct_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    # ds_nino.tct[i, 1, 1, :, :], ds_nino.tct_p[i, 1, 1, :, :] = composite_ttest(tct_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    ds_nino.vort[i, 0, 0, :, :], ds_nino.vort_p[i, 0, 0, :, :] = composite_ttest(vort_sum, event, single_sample=single)
    ds_nino.vort[i, 1, 0, :, :], ds_nino.vort_p[i, 1, 0, :, :] = composite_ttest(vort_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.vort[i, 0, 1, :, :], ds_nino.vort_p[i, 0, 1, :, :] = composite_ttest(vort_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.vort[i, 1, 1, :, :], ds_nino.vort_p[i, 1, 1, :, :] = composite_ttest(vort_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    ds_nino.dudz[i, 0, 0, :, :], ds_nino.dudz_p[i, 0, 0, :, :] = composite_ttest(dudz_sum, event, single_sample=single)
    ds_nino.dudz[i, 1, 0, :, :], ds_nino.dudz_p[i, 1, 0, :, :] = composite_ttest(dudz_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.dudz[i, 0, 1, :, :], ds_nino.dudz_p[i, 0, 1, :, :] = composite_ttest(dudz_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.dudz[i, 1, 1, :, :], ds_nino.dudz_p[i, 1, 1, :, :] = composite_ttest(dudz_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    ds_nino.omega[i, 0, 0, :, :], ds_nino.omega_p[i, 0, 0, :, :] = composite_ttest(omega_sum, event, single_sample=single)
    ds_nino.omega[i, 1, 0, :, :], ds_nino.omega_p[i, 1, 0, :, :] = composite_ttest(omega_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.omega[i, 0, 1, :, :], ds_nino.omega_p[i, 0, 1, :, :] = composite_ttest(omega_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.omega[i, 1, 1, :, :], ds_nino.omega_p[i, 1, 1, :, :] = composite_ttest(omega_win, event+np.timedelta64(2, 'Y'), single_sample=single)
    ds_nino.uwnd[i, 0, 0, :, :], ds_nino.uwnd_p[i, 0, 0, :, :] = composite_ttest(uwnd_sum, event, single_sample=single)
    ds_nino.uwnd[i, 1, 0, :, :], ds_nino.uwnd_p[i, 1, 0, :, :] = composite_ttest(uwnd_win, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.uwnd[i, 0, 1, :, :], ds_nino.uwnd_p[i, 0, 1, :, :] = composite_ttest(uwnd_sum, event+np.timedelta64(1, 'Y'), single_sample=single)
    ds_nino.uwnd[i, 1, 1, :, :], ds_nino.uwnd_p[i, 1, 1, :, :] = composite_ttest(uwnd_win, event+np.timedelta64(2, 'Y'), single_sample=single)   



In [ ]:
tct_sum_total = [[[], []], [[], []], [[], []]]
tct_win_total = [[[], []], [[], []], [[], []]]
for i in range(1, 2):
    ds = xr.open_dataset(TCT_path_grid+f'merged_0{i:02d}.nc')
    tct = ds.tct_grid.loc[:, -40:40, 0:359]
    tct, tct_sum, tct_win = process_space(tct)
    ds_sum = xr.Dataset({'tct_sum':tct_sum}, coords=tct_sum.coords)
    ds_win = xr.Dataset({'tct_win':tct_win}, coords=tct_win.coords)
    for j in range(3):
        event = events[j]
        mask_sum1 = tct_sum.time.dt.year.isin(event.astype('datetime64[Y]').astype(str).astype(int))
        mask_sum2 = tct_sum.time.dt.year.isin((event + np.timedelta64(1, 'Y')).astype('datetime64[Y]').astype(str).astype(int))
        mask_win1 = tct_win.time.dt.year.isin((event + np.timedelta64(1, 'Y')).astype('datetime64[Y]').astype(str).astype(int))
        mask_win2 = tct_win.time.dt.year.isin((event + np.timedelta64(2, 'Y')).astype('datetime64[Y]').astype(str).astype(int))
        tct_sum_total[j][0].append(tct_sum.where(mask_sum1, drop=True).values)
        tct_sum_total[j][1].append(tct_sum.where(mask_sum2, drop=True).values)
        tct_win_total[j][0].append(tct_win.where(mask_win1, drop=True).values)
        tct_win_total[j][1].append(tct_win.where(mask_win2, drop=True).values)


In [ ]:

for j in range(3):        
    ds_nino.tct[j, 0, 0, :, :], ds_nino.tct_p[j, 0, 0, :, :] = composite_tctest(np.reshape(np.array(tct_sum_total[j][0]), (-1, 81, 360)))
    ds_nino.tct[j, 0, 1, :, :], ds_nino.tct_p[j, 0, 1, :, :] = composite_tctest(np.reshape(np.array(tct_sum_total[j][1]), (-1, 81, 360)))
    ds_nino.tct[j, 1, 0, :, :], ds_nino.tct_p[j, 1, 0, :, :] = composite_tctest(np.reshape(np.array(tct_win_total[j][0]), (-1, 81, 360)))
    ds_nino.tct[j, 1, 1, :, :], ds_nino.tct_p[j, 1, 1, :, :] = composite_tctest(np.reshape(np.array(tct_win_total[j][1]), (-1, 81, 360)))
# ds_nino.to_netcdf('./significant/composite_ssta_season.nc')

In [ ]:
ds_nino.tct_p[0, 0, 0, :, :].plot.contourf(levels=21)

In [ ]:
ds_nino.tct[0, 0, 1, :, :].plot.contourf(levels=21)

In [ ]:
month = np.arange(1, 13)
nt, nx = len(month), len(lon)
ds_nino = xr.Dataset(
    {
        'ssta_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
        'p_hov':(['type', 'stage', 'month', 'lon'], np.empty((4, 3, nt, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'CP', 'EP', 'SUC']),
        'stage':(['stage'], ['current', 'next', 'next2']),
        'month':(['month'], month),
        'lon':(['lon'], lon)
    }
)

for i in range(4):
    event = events[i]
    ds_nino.ssta_hov[i, 0, :, :], ds_nino.p_hov[i, 0, :, :] = composite_ttest_time(ssta_hov, event)
    ds_nino.ssta_hov[i, 1, :, :], ds_nino.p_hov[i, 1, :, :] = composite_ttest_time(ssta_hov, event+np.timedelta64(1, 'Y'))
    ds_nino.ssta_hov[i, 2, :, :], ds_nino.p_hov[i, 2, :, :] = composite_ttest_time(ssta_hov, event+np.timedelta64(2, 'Y'))

ds_nino.to_netcdf('./significant/composite_ssta_hov.nc')


In [ ]:
80*727/1024

In [ ]:
mask = tct_win.time.dt.year.isin(cp[:].astype('datetime64[Y]').astype(str).astype(int))
tct_comp = tct_win.where(mask, drop=True)
# tar1 = tct_comp.loc[:, 0:25, 140:180]
# mask_re = tar1.mean(dim='time') > 2.5
# tc_region = tar1.where(mask_re).mean(dim=['lat', 'lon'])

In [ ]:
t, p = stats.ttest_1samp(tc_region, 0)

In [ ]:
t, p

In [ ]:
tct_comp.plot.contourf(levels=21, col='time', col_wrap=2)

In [ ]:
tct_comp.mean(dim='time').plot.contourf(levels=21)